# R Code Translation - Hardcoded and Empirical Hessian Analysis

This notebook contains code translated from R (`sharpness_plots.R`) for analyzing hardcoded and empirical Hessian statistics.

**Note**: This is legacy code. The perturbation analysis has been superseded by the more recent analysis in `analyze_perturbation_results.ipynb`.

In [ ]:
# from here and below is Paul's R code translated from cursor
!pwd

In [8]:
import osimport reimport pandas as pdimport plotly.express as pximport plotly.graph_objects as gofrom plotly.subplots import make_subplotsimport numpy as np# Translate getHardcodedStats function from R# Create sparseString column for legend labelsdef create_sparse_string(width):    return f"{int(width)}-sparse"sparse_levels = ["1-sparse", "7-sparse", "14-sparse", "20-sparse"]def get_hardcoded_stats():    """    Load and process hardcoded hessian statistics from CSV files.        Parameters:    -----------    perturbation : float        Perturbation level (0, 0.001, 0.0001, 0.00001)        Returns:    --------    pd.DataFrame        DataFrame with columns: deg, width, sharpness, norm, sd_sharpness, perturbation, type    """    # Handle path resolution - use workspace-relative paths or expanduser    base_path = os.path.expanduser("~")    workspace_path = "/dartfs/rc/lab/C/CybenkoG/new_bool_sens/boolean_function_sensitivity"            # Try workspace path first, then downloads    csv_path = os.path.join(workspace_path, "hardcoded_hessian (14).csv")            # Read CSV    sharpness_dat = pd.read_csv(csv_path, low_memory=False)        # Group by deg and width, calculate statistics    plot_dat = sharpness_dat.groupby(["deg", "width"]).agg({        "trace_train": ["mean", "std"],        "frobenius_weight_norm": "mean"    }).reset_index()        # Flatten column names    plot_dat.columns = ["deg", "width", "sharpness", "sd_sharpness", "norm"]        # Sort by deg    plot_dat = plot_dat.sort_values("deg")        # Filter to deg <= 5    plot_dat = plot_dat[plot_dat["deg"] <= 5].copy()        # Add perturbation and type columns    plot_dat["type"] = "Theoretical"        return plot_dathardcodedPlot = get_hardcoded_stats()hardcodedPlot# Define variables for plottingunique_widths = sorted(hardcodedPlot["width"].unique())colors = ["red", "brown", "green", "purple"]color_map = {width: colors[i % len(colors)] for i, width in enumerate(unique_widths)}hardcodedPlot# Plot 4: Basic hardcoded hessian plothardcodedPlot["sparseString"] = hardcodedPlot["width"].apply(create_sparse_string)hardcodedPlot["sparseString"] = pd.Categorical(hardcodedPlot["sparseString"], categories=sparse_levels, ordered=True)fig4 = go.Figure()for width in unique_widths:    subset = hardcodedPlot[hardcodedPlot["width"] == width]    if len(subset) > 0:        fig4.add_trace(            go.Scatter(                x=subset["deg"],                y=subset["sharpness"],                mode='lines+markers',                name=f"{int(width)}-sparse",                line=dict(color=color_map[width])            )        )fig4.update_layout(    title="Comparison of Hessian Trace for Empirical vs Theoretical Bound",    xaxis_title="Degree",    yaxis_title="Trace Hessian",    height=500,    width=800,    margin=dict(t=80, b=50, l=50, r=50))fig4.show()

SyntaxError: invalid syntax (353093782.py, line 1)

In [7]:
# Translate getEmpiricalStats function from Rdef get_empirical_stats(SAM=True):    """    Load and process empirical statistics from summary CSV files.        Parameters:    -----------    SAM : bool        Whether to use SAM data (True) or non-SAM data (False)        Returns:    --------    pd.DataFrame        DataFrame with columns: deg, width, sharpness, norm, error, count, type    """    base_path = os.path.expanduser("~")    workspace_path = "/dartfs/rc/lab/C/CybenkoG/new_bool_sens/boolean_function_sensitivity"        # Try downloads folder first, then workspace    csv_path1 = os.path.join(workspace_path, "HESSIAN_CALCS_102", "summary.csv")    #csv_path2 = os.path.join(workspace_path, "HESSIAN_CALCS_101", "summary.csv")            test_dat = pd.read_csv(csv_path1, low_memory=False)       #test_dat2 = pd.read_csv(csv_path2, low_memory=False)    #test_dat = pd.concat([test_dat, test_dat2], ignore_index=True)    # Remove first row (R does test_dat[-1,])    if len(test_dat) > 0:        test_dat = test_dat.iloc[1:].copy()        # Process trace column - extract mean from string like "[1,2,3]"    def parse_trace(trace_str):        if pd.isna(trace_str):            return np.nan        # Remove brackets and split by comma        trace_str = str(trace_str).strip()        if trace_str.startswith('[') and trace_str.endswith(']'):            trace_str = trace_str[1:-1]        try:            values = [float(x.strip()) for x in trace_str.split(',') if x.strip()]            return np.mean(values) if values else np.nan        except:            return np.nan        test_dat["trace"] = test_dat["trace"].apply(parse_trace)        # Process val_loss and train_loss - extract numeric values from strings    def parse_loss(loss_str):        if pd.isna(loss_str):            return np.nan        loss_str = str(loss_str)        # Remove "tensor(" prefix and ")" suffix if present        if "tensor(" in loss_str.lower():            # Extract the number after tensor(            match = re.search(r'tensor\(([^)]+)\)', loss_str, re.IGNORECASE)            if match:                return float(match.group(1))        # Try direct conversion        try:            return float(loss_str.replace(")", "").strip())        except:            return np.nan        test_dat["val_loss"] = test_dat["val_loss"].apply(parse_loss)    test_dat["train_loss"] = test_dat["train_loss"].apply(parse_loss)        # Add deg_string and gen_gap    test_dat["deg_string"] = "Degree-" + test_dat["deg"].astype(str)    test_dat["gen_gap"] = test_dat["val_loss"] - test_dat["train_loss"]        # Define split factors    split_factors = ["deg", "deg_string", "width", "func", "batch_size", "lr",                      "dropout", "wd", "d", "n_samples", "stop_loss", "f"]        # Filter by threshold    THRESHOLD = True    threshold = 0.02        if THRESHOLD:        test_dat = test_dat[test_dat["train_loss"] <= threshold].copy()        # Get final epoch (minimum epoch per split_factors)        test_dat["final_epoch"] = test_dat.groupby(split_factors)["epoch"].transform("min")    else:        test_dat["final_epoch"] = test_dat.groupby(split_factors)["epoch"].transform("max")        # Filter to final epoch    final_dat_single = test_dat[test_dat["epoch"] == test_dat["final_epoch"]].copy()        # Get minimum loss per split_factors    final_dat_single["min_loss"] = final_dat_single.groupby(split_factors)["train_loss"].transform("min")    final_dat_single = final_dat_single[final_dat_single["train_loss"] == final_dat_single["min_loss"]].copy()        # Remove duplicates - keep first occurrence per split_factors    final_dat_single = final_dat_single.drop_duplicates(subset=split_factors, keep="first").copy()        # Add converged column    final_dat_single["converged"] = final_dat_single["train_loss"] <= (threshold + 0.005)        # Filter to converged    final_dat_single = final_dat_single[final_dat_single["converged"] == True].copy()        # Define columns to keep    additional_cols = ["train_loss", "gen_gap", "time_elapsed", "epoch", "ln",                        "trace_train", "top_eig", "weight_norm", "converged"]    keep_cols = split_factors + additional_cols + ["final_epoch"]        # Select only columns that exist    available_cols = [col for col in keep_cols if col in final_dat_single.columns]    plot_dat = final_dat_single[available_cols].copy()        # Group by split_factors_wo_func    split_factors_wo_func = ["deg", "deg_string", "width", "batch_size", "lr",                               "dropout", "wd", "d", "n_samples", "stop_loss", "f"]        # Only use factors that exist in the dataframe    available_factors = [f for f in split_factors_wo_func if f in plot_dat.columns]        # Check if trace_train exists, otherwise use trace    trace_col = "trace_train" if "trace_train" in plot_dat.columns else "trace"        agg_dict = {        "gen_gap": "mean",        "weight_norm": "mean",        "top_eig": "mean"    }    if trace_col in plot_dat.columns:        agg_dict[trace_col] = "median"        plot_dat = plot_dat.groupby(available_factors).agg(agg_dict).reset_index()        # Count functions    func_counts = final_dat_single.groupby(available_factors).size().reset_index(name="func_count")    plot_dat = plot_dat.merge(func_counts, on=available_factors, how="left")        # Rename columns - use trace_col if it exists    rename_dict = {        "weight_norm": "norm",        "gen_gap": "error"    }    if trace_col in plot_dat.columns:        rename_dict[trace_col] = "sharpness"    plot_dat = plot_dat.rename(columns=rename_dict)        # Select and rename columns to match R output    # Ensure sharpness column exists (use trace if trace_train wasn't available)    if "sharpness" not in plot_dat.columns and "trace" in plot_dat.columns:        plot_dat["sharpness"] = plot_dat["trace"]        # Select columns that exist    output_cols = ["deg", "width", "sharpness", "norm", "error", "func_count"]    available_output_cols = [col for col in output_cols if col in plot_dat.columns]    plot_dat = plot_dat[available_output_cols].copy()    plot_dat = plot_dat.rename(columns={"func_count": "count"})        # Add SAM and type columns    plot_dat["SAM"] = SAM    if SAM:        plot_dat["type"] = "Empirical (SAM)"    else:        plot_dat["type"] = "Empirical"        # Sort by deg (convert to numeric first to handle mixed types)    plot_dat["deg"] = pd.to_numeric(plot_dat["deg"], errors="coerce")    plot_dat = plot_dat.sort_values("deg")        return plot_datemp_plot = get_empirical_stats(SAM=False)emp_plot_sam = get_empirical_stats(SAM=True)# Combine plotscomb_plot2 = pd.concat([hardcodedPlot, emp_plot], ignore_index=True)comb_plot3 = pd.concat([emp_plot, emp_plot_sam], ignore_index=True)comb_plot2["sparseString"] = comb_plot2["width"].apply(create_sparse_string)comb_plot3["sparseString"] = comb_plot3["width"].apply(create_sparse_string)# Convert to categorical with specified ordercomb_plot2["sparseString"] = pd.Categorical(comb_plot2["sparseString"], categories=sparse_levels, ordered=True)comb_plot3["sparseString"] = pd.Categorical(comb_plot3["sparseString"], categories=sparse_levels, ordered=True)

/scratch/d28979q/ipykernel_212068/399248358.py:25: DtypeWarning:

Columns (0,1) have mixed types. Specify dtype option on import or set low_memory=False.



TypeError: '<' not supported between instances of 'str' and 'float'

FileNotFoundError: [Errno 2] No such file or directory: '/Users/paullintilhac/code/boolean_function_sensitivity/NEURIPS_CAMERA2_NOSAM/summary.csv'

In [ ]:
# Plot 1: Comparison of Hessian Trace (Theoretical vs Empirical)
# Filter to deg <= 5
comb_plot2_filtered = comb_plot2[comb_plot2["deg"] <= 5].copy()

# Create plotly figure with subplots for each type
fig1 = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Theoretical", "Empirical"),
    horizontal_spacing=0.15
)

# Get unique widths and colors
unique_widths = sorted(comb_plot2_filtered["width"].unique())
colors = ["red", "brown", "green", "purple"]
color_map = {width: colors[i % len(colors)] for i, width in enumerate(unique_widths)}

# Plot theoretical data
theoretical_data = comb_plot2_filtered[comb_plot2_filtered["type"] == "Theoretical"]
for width in unique_widths:
    subset = theoretical_data[theoretical_data["width"] == width]
    if len(subset) > 0:
        fig1.add_trace(
            go.Scatter(
                x=subset["deg"],
                y=np.log10(subset["sharpness"]),
                mode='lines+markers',
                name=f"{int(width)}-sparse",
                line=dict(color=color_map[width]),
                showlegend=True
            ),
            row=1, col=1
        )

# Plot empirical data
empirical_data = comb_plot2_filtered[comb_plot2_filtered["type"] == "Empirical"]
for width in unique_widths:
    subset = empirical_data[empirical_data["width"] == width]
    if len(subset) > 0:
        fig1.add_trace(
            go.Scatter(
                x=subset["deg"],
                y=np.log10(subset["sharpness"]),
                mode='lines+markers',
                name=f"{int(width)}-sparse",
                line=dict(color=color_map[width]),
                showlegend=False
            ),
            row=1, col=2
        )

fig1.update_layout(
    title="Comparison of Hessian Trace for Empirical vs Theoretical Bound",
    height=500,
    width=1200,
    margin=dict(t=80, b=50, l=50, r=50)
)
fig1.update_xaxes(title_text="Degree", row=1, col=1)
fig1.update_xaxes(title_text="Degree", row=1, col=2)
fig1.update_yaxes(title_text="log10(Trace Hessian)", row=1, col=1)
fig1.update_yaxes(title_text="log10(Trace Hessian)", row=1, col=2)

fig1.show()

In [ ]:
# Plot 2: Comparison of Norm (Theoretical vs Empirical)
# Use the same filtered data
fig2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Theoretical", "Empirical"),
    horizontal_spacing=0.15
)

# Plot theoretical norm data
for width in unique_widths:
    subset = theoretical_data[theoretical_data["width"] == width]
    if len(subset) > 0:
        fig2.add_trace(
            go.Scatter(
                x=subset["deg"],
                y=subset["norm"],
                mode='lines+markers',
                name=f"{int(width)}-sparse",
                line=dict(color=color_map[width]),
                showlegend=True
            ),
            row=1, col=1
        )

# Plot empirical norm data
for width in unique_widths:
    subset = empirical_data[empirical_data["width"] == width]
    if len(subset) > 0:
        fig2.add_trace(
            go.Scatter(
                x=subset["deg"],
                y=subset["norm"],
                mode='lines+markers',
                name=f"{int(width)}-sparse",
                line=dict(color=color_map[width]),
                showlegend=False
            ),
            row=1, col=2
        )

fig2.update_layout(
    title="Comparison of Norm for Empirical vs Theoretical Bound",
    height=500,
    width=1200,
    margin=dict(t=80, b=50, l=50, r=50)
)
fig2.update_xaxes(title_text="Degree", row=1, col=1)
fig2.update_xaxes(title_text="Degree", row=1, col=2)
fig2.update_yaxes(title_text="Frobenius Norm", row=1, col=1)
fig2.update_yaxes(title_text="Frobenius Norm", row=1, col=2)

fig2.show()

In [ ]:
# # Plot 3: Perturbation effects on Theoretical Hessian
# # Create perturbation string labels
# def create_pert_string(pert):
#     if pert == 0:
#         return "σ = 0"
#     elif pert == 0.00001:
#         return "σ = 1e-05"
#     elif pert == 0.0001:
#         return "σ = 1e-04"
#     elif pert == 0.001:
#         return "σ = 0.001"
#     else:
#         return f"σ = {pert}"

# comb_plot1["pert_string"] = comb_plot1["perturbation"].apply(create_pert_string)
# pert_levels = ["σ = 0", "σ = 1e-05", "σ = 1e-04", "σ = 0.001"]
# comb_plot1["pert_string"] = pd.Categorical(comb_plot1["pert_string"], categories=pert_levels, ordered=True)

# # Calculate error bars
# comb_plot1["upper"] = comb_plot1["sharpness"] + 1 * comb_plot1["sd_sharpness"]
# comb_plot1["lower"] = comb_plot1["sharpness"] - 1 * comb_plot1["sd_sharpness"]

# # Filter to deg <= 4
# comb_plot1_filtered = comb_plot1[comb_plot1["deg"] <= 4].copy()

# # Get unique perturbation levels for faceting
# unique_perts = sorted(comb_plot1_filtered["perturbation"].unique())
# num_perts = len(unique_perts)
# num_cols = 2
# num_rows = (num_perts + num_cols - 1) // num_cols

# fig3 = make_subplots(
#     rows=num_rows, cols=num_cols,
#     subplot_titles=[create_pert_string(p) for p in unique_perts],
#     horizontal_spacing=0.15,
#     vertical_spacing=0.15
# )

# # Plot for each perturbation level
# for idx, pert in enumerate(unique_perts):
#     row = idx // num_cols + 1
#     col = idx % num_cols + 1
    
#     pert_data = comb_plot1_filtered[comb_plot1_filtered["perturbation"] == pert]
    
#     for width in unique_widths:
#         subset = pert_data[pert_data["width"] == width]
#         if len(subset) > 0:
#             # Main line with error bars
#             fig3.add_trace(
#                 go.Scatter(
#                     x=subset["deg"],
#                     y=subset["sharpness"],
#                     mode='lines+markers',
#                     name=f"{int(width)}-sparse",
#                     line=dict(color=color_map[width]),
#                     error_y=dict(
#                         type='data',
#                         symmetric=False,
#                         array=subset["upper"] - subset["sharpness"],
#                         arrayminus=subset["sharpness"] - subset["lower"],
#                         width=5,
#                         thickness=1.5
#                     ),
#                     showlegend=(idx == 0)  # Only show legend for first subplot
#                 ),
#                 row=row, col=col
#             )

# fig3.update_layout(
#     title="Plot of Hessian Trace of Construction with Increasing Perturbations",
#     height=300 * num_rows,
#     width=1200,
#     margin=dict(t=80, b=50, l=50, r=50)
# )

# # Update axes
# for row in range(1, num_rows + 1):
#     for col in range(1, num_cols + 1):
#         fig3.update_xaxes(title_text="Degree", row=row, col=col)
#         fig3.update_yaxes(title_text="Hessian Trace", row=row, col=col, range=[-100000, 2000000])

# fig3.show()